# Stage 7: Final evaluation and error analysis
**Deep Learning-Based Detection and Classification of Brain Tumors Using MRI Images**

This notebook reads the saved predictions of all 20 training runs (no model is trained here, no GPU needed) and produces the final tables, figures and error analysis. Every number is recomputed from the saved class probabilities and checked against the experiment table.

**Inputs you need**
1. `final_results.zip` (merged results of Stages 3 to 6, provided with this notebook). Upload it when asked.
2. The dataset (same options as before: Kaggle or a zip in Google Drive). It is used only to draw example images and to compute image similarity.

**Outputs:** `stage7_results.zip` with `tables/` and `figures/`.

**Scope note:** the selection of the final model uses validation results and the rule fixed before Stage 6. Test results are reported, not used to choose.

## 1. Scripts

In [ ]:
import os, zipfile, subprocess, sys
from pathlib import Path
W = Path('/content/stage7'); W.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable,'-m','pip','-q','install','scipy','seaborn','scikit-learn'], check=False)

In [ ]:
%%writefile /content/stage7/final_evaluation.py
"""Stage 7: final evaluation and error analysis.

Reads ONLY saved results (prediction files with class probabilities, training histories, the experiment table,
the locked split table) plus the MRI images for the example figures. It needs no GPU and no PyTorch.
Every number written to disk is computed here from those files.

Usage:
    python final_evaluation.py --results final_results --data-root <folder containing Train/ and Val/> --src <project src folder> --out stage7_results
"""
import argparse, json, re, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image
from scipy.stats import fisher_exact
from sklearn.metrics import roc_curve, auc, classification_report

CL = ['Glioma', 'Meningioma', 'No Tumor', 'Pituitary']
# Class colours: first four slots of the validated categorical palette (blue, orange, aqua, yellow).
# Aqua and yellow are below 3:1 contrast on white, so every figure also uses direct labels / line styles / a CSV table.
CLASS_COL = {'Glioma': '#2a78d6', 'Meningioma': '#eb6834', 'No Tumor': '#1baf7a', 'Pituitary': '#eda100'}
FAMILY_COL = {'CNN from scratch': '#8a8985', 'ResNet18': '#2a78d6', 'ResNet50': '#eb6834'}
INK, INK2, GRID = '#0b0b0b', '#52514e', '#e4e3df'
FULL = {'ResNet18': ['R18-C', 'R18-C-s43', 'R18-C-s44'], 'ResNet50': ['R50-C', 'F-R50-s43', 'F-R50-s44']}
FINAL = 'R18-C'
CONFIGS = [('CNN from scratch', 'CNN from scratch', ['B1']),
           ('CNN, weighted loss', 'CNN from scratch', ['B2']),
           ('ResNet18, frozen backbone', 'ResNet18', ['R18-A']),
           ('ResNet18, layer4 + fc', 'ResNet18', ['R18-B']),
           ('ResNet18, full fine-tuning', 'ResNet18', FULL['ResNet18']),
           ('ResNet50, frozen backbone', 'ResNet50', ['R50-A']),
           ('ResNet50, layer4 + fc', 'ResNet50', ['R50-B']),
           ('ResNet50, full fine-tuning', 'ResNet50', FULL['ResNet50'])]
TUNING = ['T1-lr1e-5', 'T2-lr1e-3', 'T3-bs16', 'T4-adamw', 'T5-dropout', 'T6-weighted', 'T7-hflip']


# ----------------------------------------------------------------------------------------------- helpers
def style():
    plt.rcParams.update({'font.size': 9.5, 'axes.edgecolor': INK2, 'axes.labelcolor': INK, 'text.color': INK, 'xtick.color': INK2,
                         'ytick.color': INK2, 'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True,
                         'grid.color': GRID, 'grid.linewidth': .8, 'axes.axisbelow': True, 'figure.facecolor': 'white',
                         'axes.facecolor': 'white', 'savefig.facecolor': 'white', 'legend.frameon': False})


def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d; h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h


def load_pred(res, exp, split):
    p = pd.read_csv(res / 'predictions' / f'{exp}_{split}_predictions.csv').set_index('rel').sort_index()
    y = p.true_label.map({c: i for i, c in enumerate(CL)}).values
    P = p[[f'prob_{c}' for c in CL]].values
    return p, y, P


def savefig(fig, path):
    fig.savefig(path, dpi=200, bbox_inches='tight'); plt.close(fig)


def heatmap(ax, cm, title):
    cm = np.asarray(cm); norm = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    ax.imshow(norm, cmap='Blues', vmin=0, vmax=1)
    for i in range(4):
        for j in range(4):
            col = 'white' if norm[i, j] > .55 else INK
            ax.text(j, i, f'{cm[i, j]}\n{norm[i, j] * 100:.1f}%', ha='center', va='center', color=col, fontsize=9)
    ax.set_xticks(range(4)); ax.set_xticklabels(CL, rotation=20, ha='right'); ax.set_yticks(range(4)); ax.set_yticklabels(CL)
    ax.set_xlabel('Predicted class'); ax.set_ylabel('True class'); ax.set_title(title, fontsize=10); ax.grid(False)
    for s in ax.spines.values(): s.set_visible(False)


def draw_img(ax, root, rel, row, show_box, title):
    im = Image.open(root / rel).convert('L'); W, H = im.size
    ax.imshow(im, cmap='gray', vmin=0, vmax=255); ax.axis('off')
    if show_box:
        for b in json.loads(row.boxes_json):
            _, cx, cy, w, h = b
            ax.add_patch(Rectangle(((cx - w / 2) * W, (cy - h / 2) * H), w * W, h * H, fill=False, edgecolor='#ffd23f', lw=1.3, ls='--'))
    ax.set_title(title, fontsize=7.8, color=INK, pad=3)


def source_style(stem):
    """Filename pattern as a rough proxy for the original source of an image (dataset appears to merge several collections)."""
    if re.match(r'^(Tr|Te)-', stem): return 'Tr-/Te- prefixed'
    if re.match(r'^gg\s*\(', stem): return 'gg (n)'
    if re.match(r'^p\s*\(', stem): return 'p (n)'
    if re.match(r'^m\d?\s*\(', stem): return 'm (n) / m1-3 (n)'
    if re.match(r'^image\s*\(', stem): return 'image (n)'
    if re.match(r'^\d+$', stem): return 'number only'
    return 'other'


# ----------------------------------------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--results', required=True); ap.add_argument('--data-root', required=True)
    ap.add_argument('--src', required=True); ap.add_argument('--out', required=True)
    a = ap.parse_args()
    res, root, out = Path(a.results), Path(a.data_root), Path(a.out)
    (out / 'tables').mkdir(parents=True, exist_ok=True); (out / 'figures').mkdir(exist_ok=True)
    sys.path.insert(0, a.src)
    from evaluate import compute_metrics, bootstrap_ci   # pure numpy/sklearn functions from the project
    import visualization as vz
    style()
    table = pd.read_csv(res / 'metrics' / 'experiment_table_master.csv').set_index('experiment_id')
    splits = pd.read_csv(res / 'metrics' / 'splits.csv').set_index('rel')
    S = {}   # summary numbers for the report

    # ---- 1. metrics for every run, recomputed from the saved probabilities
    M = {}
    rows = []
    for e in table.index:
        r = {'experiment_id': e}
        for split in ['val', 'test']:
            p, y, P = load_pred(res, e, split)
            m = compute_metrics(y, P, CL); M[(e, split)] = (m, y, P)
            s = m['summary']
            r.update({f'{split}_{k}': s[k] for k in ['accuracy', 'balanced_accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'f1_weighted', 'roc_auc_ovr_macro']})
        rows.append(r)
    allruns = pd.DataFrame(rows).set_index('experiment_id')
    for c, tc in [('val_accuracy', 'val_accuracy'), ('test_accuracy', 'test_accuracy'), ('val_f1_macro', 'val_macro_f1'), ('test_f1_macro', 'test_macro_f1')]:
        assert np.allclose(allruns[c], table.loc[allruns.index, tc], atol=1e-6), f'recomputed {c} differs from the experiment table'
    allruns = allruns.join(table[['model', 'finetune_strategy', 'seed', 'best_epoch', 'epochs_run', 'trainable_params', 'train_time_s']])
    allruns.round(4).to_csv(out / 'tables' / 'all_runs_metrics.csv')
    S['recomputed_metrics_match_experiment_table'] = True

    # ---- 2. model comparison table (the table requested in the project brief)
    def agg(ids, split, col):
        v = allruns.loc[ids, f'{split}_{col}'].values
        return v.mean(), (v.std(ddof=1) if len(v) > 1 else np.nan)
    cols = [('accuracy', 'Accuracy'), ('precision_macro', 'Precision (macro)'), ('recall_macro', 'Recall (macro)'), ('f1_macro', 'F1 (macro)'),
            ('roc_auc_ovr_macro', 'ROC-AUC (OvR macro)'), ('balanced_accuracy', 'Balanced accuracy')]
    spec = [('CNN baseline (B1), single run', ['B1']), ('ResNet18, full fine-tuning, mean of 3 seeds', FULL['ResNet18']),
            ('ResNet50, full fine-tuning, mean of 3 seeds', FULL['ResNet50']),
            ('Final model: ResNet18 full fine-tuning, seed-42 run (R18-C)', [FINAL])]
    num, txt = [], []
    for name, ids in spec:
        rn, rt = {'model': name, 'runs': ' + '.join(ids)}, {'Model': name}
        for k, lab in cols:
            mu, sd = agg(ids, 'test', k); rn[lab + ' mean'] = mu; rn[lab + ' std'] = sd
            rt[lab] = f'{mu:.4f} ± {sd:.4f}' if len(ids) > 1 else f'{mu:.4f}'
        num.append(rn); txt.append(rt)
    pd.DataFrame(num).round(4).to_csv(out / 'tables' / 'model_comparison_numeric.csv', index=False)
    pd.DataFrame(txt).to_csv(out / 'tables' / 'model_comparison_table.csv', index=False)
    S['model_comparison'] = txt
    ext = []
    for label, fam, ids in CONFIGS:
        r = {'configuration': label, 'runs': len(ids), 'run_ids': ' '.join(ids), 'trainable_params_M': round(table.loc[ids[0], 'trainable_params'] / 1e6, 2),
             'mean_epochs_run': round(table.loc[ids, 'epochs_run'].mean(), 1), 'mean_minutes_per_run': round(table.loc[ids, 'train_time_s'].mean() / 60, 1)}
        for split in ['val', 'test']:
            for k in ['accuracy', 'f1_macro']:
                mu, sd = agg(ids, split, k); r[f'{split}_{k}'] = f'{mu:.4f} ± {sd:.4f}' if len(ids) > 1 else f'{mu:.4f}'
        mu, sd = agg(ids, 'test', 'roc_auc_ovr_macro'); r['test_roc_auc'] = f'{mu:.4f} ± {sd:.4f}' if len(ids) > 1 else f'{mu:.4f}'
        ext.append(r)
    pd.DataFrame(ext).to_csv(out / 'tables' / 'all_configurations_summary.csv', index=False)

    # ---- 3. final model: report, per-class metrics with intervals
    mf, yf, Pf = M[(FINAL, 'test')]
    ci = bootstrap_ci(yf, Pf.argmax(1), n_boot=2000, seed=42)
    pc = mf['per_class'].copy(); cm = mf['confusion_matrix']
    pc['recall_95CI_low'], pc['recall_95CI_high'] = zip(*[wilson(cm[i, i], cm[i].sum()) for i in range(4)])
    pc.round(4).to_csv(out / 'tables' / 'final_model_per_class_test.csv', index=False)
    (out / 'tables' / 'final_model_classification_report_test.txt').write_text(classification_report(yf, Pf.argmax(1), target_names=CL, digits=4))
    S['final_model_test'] = {**{k: round(v, 4) for k, v in mf['summary'].items() if k != 'n'}, 'n': mf['summary']['n'], 'bootstrap_ci_accuracy': ci['accuracy_ci'], 'bootstrap_ci_macroF1': ci['f1_macro_ci']}
    S['final_model_per_class_recall'] = {c: round(float(pc.set_index('cls').loc[c, 'recall_sensitivity']), 4) for c in CL}
    pcs = []
    for name, ids in [('CNN baseline (B1)', ['B1'])] + [(k, v) for k, v in FULL.items()]:
        d = pd.concat([M[(e, 'test')][0]['per_class'].set_index('cls')[['precision', 'recall_sensitivity', 'specificity', 'f1', 'roc_auc_ovr']] for e in ids], keys=ids)
        g = d.groupby(level=1, sort=False).agg(['mean', 'std']); g.columns = [f'{a}_{b}' for a, b in g.columns]; g.insert(0, 'model', name); pcs.append(g.reset_index().rename(columns={'cls': 'class'}))
    pd.concat(pcs).round(4).to_csv(out / 'tables' / 'per_class_by_model_test.csv', index=False)

    # ---- 4. confusion matrices
    fig, ax = plt.subplots(1, 2, figsize=(10.5, 4.6))
    heatmap(ax[0], cm, f'Final model (ResNet18, seed 42), test set, n={len(yf)}')
    heatmap(ax[1], M[('B1', 'test')][0]['confusion_matrix'], 'CNN baseline (B1), test set')
    fig.tight_layout(); savefig(fig, out / 'figures' / 'confusion_matrix_final_vs_baseline.png')
    pooled = {k: sum(M[(e, 'test')][0]['confusion_matrix'] for e in ids) for k, ids in FULL.items()}
    fig, ax = plt.subplots(1, 2, figsize=(10.5, 4.6))
    for a_, (k, c_) in zip(ax, pooled.items()): heatmap(a_, c_, f'{k}, 3 seeds pooled (counts are sums over 3 runs)')
    fig.tight_layout(); savefig(fig, out / 'figures' / 'confusion_matrix_pooled_3_seeds.png')
    for k, c_ in pooled.items(): pd.DataFrame(c_, index=CL, columns=CL).to_csv(out / 'tables' / f'confusion_matrix_pooled_{k}.csv')
    pd.DataFrame(cm, index=CL, columns=CL).to_csv(out / 'tables' / 'confusion_matrix_final_model_test.csv')

    # ---- 5. ROC curves, final model
    fig, ax = plt.subplots(1, 2, figsize=(10.5, 4.8)); ls = {'Glioma': '-', 'Meningioma': '--', 'No Tumor': '-.', 'Pituitary': ':'}
    for a_, zoom in zip(ax, [False, True]):
        for k, c in enumerate(CL):
            fpr, tpr, _ = roc_curve((yf == k).astype(int), Pf[:, k])
            a_.plot(fpr, tpr, color=CLASS_COL[c], ls=ls[c], lw=2, label=f'{c} (AUC {auc(fpr, tpr):.4f})')
        if not zoom: a_.plot([0, 1], [0, 1], color=INK2, lw=1, ls=(0, (1, 3))); a_.set_title('One-vs-rest ROC, final model, test set')
        else: a_.set_xlim(0, .1); a_.set_ylim(.8, 1.005); a_.set_title('Zoom: false positive rate 0 to 0.1')
        a_.set_xlabel('False positive rate'); a_.set_ylabel('True positive rate')
    ax[0].legend(loc='lower right', fontsize=8.5)
    fig.tight_layout(); savefig(fig, out / 'figures' / 'roc_final_model.png')

    # ---- 6. model comparison chart (each dot is one training run)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.6), sharey=True); ypos = np.arange(len(CONFIGS))[::-1]
    for a_, split, title in zip(ax, ['test', 'val'], ['Test macro F1', 'Validation macro F1']):
        for (label, fam, ids), y0 in zip(CONFIGS, ypos):
            v = allruns.loc[ids, f'{split}_f1_macro'].values; col = FAMILY_COL[fam]
            a_.plot([v.min(), v.max()], [y0, y0], color=col, lw=2.2, alpha=.45, solid_capstyle='round')
            off = np.linspace(-.16, .16, len(v)) if len(v) > 1 else np.zeros(1)
            a_.scatter(v, y0 + off, s=46, color=col, edgecolor='white', linewidth=1.5, zorder=3)
            a_.text(v.max() + .004, y0, f'{v.mean():.3f}' + (f' (mean of {len(v)})' if len(v) > 1 else ''), va='center', fontsize=8.3, color=INK2)
        a_.set_title(title); a_.set_xlim(.86, 1.045); a_.set_xticks(np.arange(.86, 1.001, .02)); a_.set_xlabel('Macro F1')
        a_.grid(axis='y', visible=False)
    ax[0].set_yticks(ypos); ax[0].set_yticklabels([c[0] for c in CONFIGS])
    fig.suptitle('Each dot is one training run; lines span the 3 seeds where seeds were repeated', fontsize=9.5, color=INK2, y=1.0)
    fig.tight_layout(); savefig(fig, out / 'figures' / 'model_comparison_macro_f1.png')

    # ---- 7. tuning chart (Stage 6a), validation only
    ref_ids = FULL['ResNet18']; ref = allruns.loc[ref_ids, 'val_f1_macro']; thr = ref.mean() + max(ref.max() - ref.min(), .005)
    names = {'T1-lr1e-5': 'T1 pretrained-layer lr 1e-5', 'T2-lr1e-3': 'T2 pretrained-layer lr 1e-3', 'T3-bs16': 'T3 batch size 16', 'T4-adamw': 'T4 AdamW, weight decay 0.01',
             'T5-dropout': 'T5 dropout 0.3', 'T6-weighted': 'T6 class-weighted loss', 'T7-hflip': 'T7 horizontal flip'}
    fig, ax = plt.subplots(figsize=(8.6, 4.4)); labels = ['Reference (3 seeds)', 'Reference, Stage 4 run'] + [names[t] for t in TUNING]; yy = np.arange(len(labels))[::-1]
    ax.scatter(ref.values, np.full(3, yy[0]), s=46, color='#2a78d6', edgecolor='white', linewidth=1.5, zorder=3)
    ax.scatter([allruns.loc['R18-C-orig', 'val_f1_macro']], [yy[1]], s=46, facecolor='white', edgecolor='#2a78d6', linewidth=1.8, zorder=3)
    for t, y0 in zip(TUNING, yy[2:]): ax.scatter([allruns.loc[t, 'val_f1_macro']], [y0], s=46, color='#eb6834', edgecolor='white', linewidth=1.5, zorder=3)
    ax.axvline(ref.mean(), color='#2a78d6', lw=1.2, ls='--'); ax.axvline(thr, color=INK2, lw=1.2, ls=':')
    ax.text(ref.mean() - .0008, yy[-1] - .75, f'reference mean {ref.mean():.4f}', ha='right', fontsize=8, color='#2a78d6')
    ax.text(thr + .0004, yy[-1] - .75, f'adoption threshold {thr:.4f}', ha='left', fontsize=8, color=INK2)
    ax.set_yticks(yy); ax.set_yticklabels(labels); ax.set_xlim(.968, 1.0); ax.set_ylim(yy[-1] - 1.2, yy[0] + .6); ax.set_xlabel('Validation macro F1 (563 images; 1 image is about 0.002)')
    ax.set_title('Stage 6a: no single change reached the pre-set adoption threshold'); ax.grid(axis='y', visible=False)
    fig.tight_layout(); savefig(fig, out / 'figures' / 'tuning_validation_macro_f1.png')

    # ---- 8. training curves
    hist = {e: pd.read_csv(res / 'metrics' / f'{e}_history.csv') for e in ['B1', 'R18-C', 'R50-C']}
    cols_ = {'B1': FAMILY_COL['CNN from scratch'], 'R18-C': FAMILY_COL['ResNet18'], 'R50-C': FAMILY_COL['ResNet50']}
    lab_ = {'B1': 'CNN from scratch (B1)', 'R18-C': 'ResNet18 full (R18-C)', 'R50-C': 'ResNet50 full (R50-C)'}
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
    for e, h in hist.items():
        b = int(h.val_macro_f1.idxmax())
        ax[0].plot(h.epoch, h.val_loss, color=cols_[e], lw=1.8, label=lab_[e]); ax[1].plot(h.epoch, h.val_macro_f1, color=cols_[e], lw=1.8, label=lab_[e])
        ax[0].scatter([h.epoch[b]], [h.val_loss[b]], color=cols_[e], edgecolor='white', s=55, zorder=3); ax[1].scatter([h.epoch[b]], [h.val_macro_f1[b]], color=cols_[e], edgecolor='white', s=55, zorder=3)
    ax[0].set_ylabel('Validation loss'); ax[1].set_ylabel('Validation macro F1'); ax[1].set_ylim(.4, 1.0)
    for a_ in ax: a_.set_xlabel('Epoch')
    ax[0].set_title('Validation loss (dot = epoch of the saved checkpoint)'); ax[1].set_title('Validation macro F1'); ax[1].legend(loc='lower right')
    fig.tight_layout(); savefig(fig, out / 'figures' / 'training_curves_comparison.png')
    h = hist['R18-C']; fb = int(h.val_macro_f1.idxmax())
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
    ax[0].plot(h.epoch, h.train_loss, color='#2a78d6', lw=1.8, label='Training loss'); ax[0].plot(h.epoch, h.val_loss, color='#eb6834', lw=1.8, label='Validation loss')
    ax[1].plot(h.epoch, h.train_acc, color='#2a78d6', lw=1.8, label='Training accuracy (augmented batches)'); ax[1].plot(h.epoch, h.val_acc, color='#eb6834', lw=1.8, label='Validation accuracy')
    for a_, t_ in zip(ax, ['Loss', 'Accuracy']):
        a_.axvline(h.epoch[fb], color=INK2, ls=':', lw=1.2, label=f'Saved checkpoint (epoch {int(h.epoch[fb])})'); a_.set_xlabel('Epoch'); a_.set_title(f'Final model (R18-C): {t_}'); a_.legend()
    ax[0].set_ylabel('Cross-entropy loss'); ax[1].set_ylabel('Accuracy')
    fig.tight_layout(); savefig(fig, out / 'figures' / 'training_curves_final_model.png')

    # ---- 9. error analysis
    idx = load_pred(res, FINAL, 'test')[0].index
    meta = splits.loc[idx].copy()
    runs6 = FULL['ResNet18'] + FULL['ResNet50']
    wrong = pd.DataFrame({e: ~load_pred(res, e, 'test')[0].loc[idx, 'correct'].values for e in runs6 + ['B1']}, index=idx)
    meta['n_wrong_6'] = wrong[runs6].sum(1).values; meta['wrong_final'] = wrong[FINAL].values; meta['wrong_B1'] = wrong['B1'].values
    meta['group_size'] = meta.groupby('group').group.transform('size') if 'group' in meta else 1
    gsz = splits.groupby('group').size(); meta['group_size'] = meta.group.map(gsz).values
    meta['tumour_box_area'] = [max([b[3] * b[4] for b in json.loads(j)], default=np.nan) if l != 'No Tumor' else np.nan for j, l in zip(meta.boxes_json, meta.label)]
    meta['size_group'] = np.where((meta.width == 512) & (meta.height == 512), '512x512', 'other size')
    meta['source_style'] = meta.stem.map(source_style)
    pf = load_pred(res, FINAL, 'test')[0].loc[idx]
    meta['pred_final'] = pf.pred_label.values; meta['p_pred_final'] = pf[[f'prob_{c}' for c in CL]].max(axis=1).values
    meta['p_true_final'] = [pf.loc[i, f'prob_{t}'] for i, t in zip(idx, meta.label)]
    err = meta[meta.wrong_final].copy()
    keep = ['label', 'pred_final', 'p_pred_final', 'p_true_final', 'n_wrong_6', 'wrong_B1', 'width', 'height', 'std', 'mean', 'tumour_box_area', 'group_size', 'source_style']
    err[keep].rename(columns={'label': 'true_label', 'pred_final': 'predicted', 'std': 'contrast_std', 'mean': 'mean_intensity'}).round(3).to_csv(out / 'tables' / 'misclassified_final_model_test.csv')
    S['final_model_test_errors'] = int(meta.wrong_final.sum())
    S['final_errors_by_pair'] = {f'{t} -> {p}': int(n) for (t, p), n in err.groupby(['label', 'pred_final']).size().items()}
    S['final_errors_also_wrong_in_B1'] = int(err.wrong_B1.sum())
    S['images_wrong_in_all_6_runs'] = int((meta.n_wrong_6 == 6).sum()); S['images_wrong_in_at_least_1_of_6_runs'] = int((meta.n_wrong_6 >= 1).sum())
    S['images_wrong_in_5_or_6_runs'] = int((meta.n_wrong_6 >= 5).sum())

    # pooled confusions over the six full fine-tuning runs
    pool6 = sum(M[(e, 'test')][0]['confusion_matrix'] for e in runs6)
    conf = [(CL[i], CL[j], int(pool6[i, j]), pool6[i, j] / 6) for i in range(4) for j in range(4) if i != j and pool6[i, j] > 0]
    conf = pd.DataFrame(conf, columns=['true', 'predicted', 'count_over_6_runs', 'mean_per_run']).sort_values('count_over_6_runs', ascending=False)
    conf.round(2).to_csv(out / 'tables' / 'top_confusions_6_runs.csv', index=False)
    S['top_confusions_6_runs'] = conf.head(6).round(2).to_dict('records')

    # error rate by factor (pooled over the six runs; descriptive)
    fr = []
    def add(factor, level, mask):
        sub = meta[mask]
        if len(sub): fr.append(dict(factor=factor, level=level, images=len(sub), mean_error_rate_6_runs=round(sub.n_wrong_6.sum() / (6 * len(sub)), 4), images_wrong_in_any_run=int((sub.n_wrong_6 > 0).sum()), images_wrong_in_final_model=int(sub.wrong_final.sum())))
    for c in CL: add('true class', c, meta.label == c)
    for g in ['512x512', 'other size']: add('image size', g, meta.size_group == g)
    for s_ in sorted(meta.source_style.unique()): add('filename pattern (proxy for source)', s_, meta.source_style == s_)
    tum = meta.label != 'No Tumor'; qs = meta.loc[tum, 'tumour_box_area'].quantile([1 / 3, 2 / 3]).values
    add(f'box area, tumour classes (edges {qs[0]:.3f}, {qs[1]:.3f} of image)', 'smallest third', tum & (meta.tumour_box_area <= qs[0]))
    add(f'box area, tumour classes (edges {qs[0]:.3f}, {qs[1]:.3f} of image)', 'middle third', tum & (meta.tumour_box_area > qs[0]) & (meta.tumour_box_area <= qs[1]))
    add(f'box area, tumour classes (edges {qs[0]:.3f}, {qs[1]:.3f} of image)', 'largest third', tum & (meta.tumour_box_area > qs[1]))
    cq = meta['std'].quantile([1 / 3, 2 / 3]).values
    add('contrast (pixel std) tercile', 'lowest third', meta['std'] <= cq[0]); add('contrast (pixel std) tercile', 'middle third', (meta['std'] > cq[0]) & (meta['std'] <= cq[1])); add('contrast (pixel std) tercile', 'highest third', meta['std'] > cq[1])
    add('near-duplicate group', 'singleton image', meta.group_size == 1); add('near-duplicate group', 'member of a multi-image group', meta.group_size > 1)
    fr = pd.DataFrame(fr); fr.to_csv(out / 'tables' / 'error_rate_by_factor.csv', index=False)
    def fisher(mask_a, mask_b, name):
        A, B = meta[mask_a], meta[mask_b]; t = [[int((A.n_wrong_6 > 0).sum()), int((A.n_wrong_6 == 0).sum())], [int((B.n_wrong_6 > 0).sum()), int((B.n_wrong_6 == 0).sum())]]
        return dict(comparison=name, group_a_wrong_any_vs_never=t[0], group_b_wrong_any_vs_never=t[1], fisher_exact_p=round(float(fisher_exact(t)[1]), 4))
    tests = [fisher(meta.size_group == '512x512', meta.size_group == 'other size', '512x512 vs other size (image wrong in any of 6 runs)'),
             fisher(tum & (meta.tumour_box_area <= qs[0]), tum & (meta.tumour_box_area > qs[1]), 'tumour box area: smallest vs largest third'),
             fisher(meta.group_size == 1, meta.group_size > 1, 'singleton vs multi-image group')]
    pd.DataFrame(tests).to_csv(out / 'tables' / 'error_factor_tests_exploratory.csv', index=False); S['exploratory_tests'] = tests

    # confidence of correct vs wrong predictions, pooled over six runs
    cc, cw = [], []
    for e in runs6:
        p, y, P = load_pred(res, e, 'test'); top = P.max(1); ok = P.argmax(1) == y; cc += list(top[ok]); cw += list(top[~ok])
    S['confidence_6_runs'] = dict(mean_top_prob_correct=round(float(np.mean(cc)), 4), mean_top_prob_wrong=round(float(np.mean(cw)), 4), wrong_predictions=len(cw), wrong_with_top_prob_above_0_9=int(np.sum(np.array(cw) > .9)))

    # example figures
    def grid(df, path, title, ncol=4, box=True, caption=None):
        n = len(df); nrow = int(np.ceil(n / ncol)); fig, axes = plt.subplots(nrow, ncol, figsize=(3.0 * ncol, 3.25 * nrow), squeeze=False)
        for a_ in axes.ravel(): a_.axis('off')
        for a_, (rel, r) in zip(axes.ravel(), df.iterrows()):
            if r.wrong_final: ttl = f'{r.label} → {r.pred_final}\np = {r.p_pred_final:.2f} | wrong in {int(r.n_wrong_6)}/6 runs'
            elif r.n_wrong_6 > 0: ttl = f'{r.label}: final model correct\np = {r.p_pred_final:.2f} | wrong in {int(r.n_wrong_6)}/6 runs'
            else: ttl = f'{r.label} (correct)\np = {r.p_pred_final:.2f}'
            draw_img(a_, root, rel, r, box and r.label != 'No Tumor', ttl)
        fig.suptitle(title, fontsize=10.5, y=1.0 + .02 * (3 / max(nrow, 1)))
        if caption: fig.text(.5, -.01, caption, ha='center', fontsize=8, color=INK2)
        fig.tight_layout(h_pad=2.6, w_pad=1.0); savefig(fig, path)
    cap = 'Dashed yellow boxes are the dataset annotations (tumour classes only). They are shown for context and were not used for training.'
    grid(err.sort_values(['label', 'pred_final']), out / 'figures' / 'misclassified_examples_final_model.png', f'All {len(err)} test images misclassified by the final model', caption=cap)
    hard = meta[meta.n_wrong_6 >= 5].sort_values(['label', 'n_wrong_6'])
    if len(hard): grid(hard.head(12), out / 'figures' / 'hard_examples_wrong_in_5_or_6_runs.png', f'Test images misclassified in at least 5 of the 6 ResNet runs (final-model prediction shown)', caption=cap)
    rng = np.random.default_rng(42); ok = meta[~meta.wrong_final]; pick = []
    for c in CL: pick.append(ok[ok.label == c].sample(1, random_state=int(rng.integers(0, 10**6))))
    grid(pd.concat(pick), out / 'figures' / 'correct_examples_final_model.png', 'Correctly classified test images (one random image per class, not hand-picked)', caption=cap)

    # ---- 10. figure: error rate by factor
    sub = fr[fr.factor.isin(['true class', 'image size', 'near-duplicate group'])]
    fig, ax = plt.subplots(figsize=(7.4, 3.4)); yy = np.arange(len(sub))[::-1]
    ax.barh(yy, sub.mean_error_rate_6_runs * 100, color='#2a78d6', height=.62)
    for y0, (_, r) in zip(yy, sub.iterrows()): ax.text(r.mean_error_rate_6_runs * 100 + .08, y0, f'{r.mean_error_rate_6_runs * 100:.2f}%  (n={r.images})', va='center', fontsize=8.3, color=INK2)
    ax.set_yticks(yy); ax.set_yticklabels([f'{f.split(" (")[0]}: {l}' for f, l in zip(sub.factor, sub.level)]); ax.set_xlabel('Mean test error rate over the 6 ResNet runs (%)'); ax.grid(axis='y', visible=False); ax.set_xlim(0, sub.mean_error_rate_6_runs.max() * 100 * 1.5)
    ax.set_title('Where the ResNet errors fall (descriptive; small counts)')
    fig.tight_layout(); savefig(fig, out / 'figures' / 'error_rate_by_factor.png')

    # ---- 11. cost
    cost = pd.DataFrame([{'configuration': l, 'trainable_params_M': round(table.loc[i[0], 'trainable_params'] / 1e6, 2), 'mean_epochs': round(table.loc[i, 'epochs_run'].mean(), 1),
                          'mean_minutes_per_run': round(table.loc[i, 'train_time_s'].mean() / 60, 1), 'mean_sec_per_epoch': round((table.loc[i, 'train_time_s'] / table.loc[i, 'epochs_run']).mean(), 1)} for l, f, i in CONFIGS])
    cost.to_csv(out / 'tables' / 'computational_cost.csv', index=False)
    json.dump(S, open(out / 'stage7_summary.json', 'w'), indent=2, default=str)
    print(json.dumps(S, indent=1, default=str)[:6000])
    print('\nwritten:', sorted(p.name for p in out.rglob('*') if p.is_file()))


if __name__ == '__main__':
    main()

In [ ]:
%%writefile /content/stage7/neighbour_check.py
"""Nearest-training-image similarity for every validation/test image (leakage / heterogeneity diagnostic).
Similarity = Pearson correlation of 64x64 grayscale thumbnails. Uses only the images and the locked split."""
import sys, json, re, numpy as np, pandas as pd
from pathlib import Path
from PIL import Image
from scipy.stats import mannwhitneyu, fisher_exact
res, root, out = Path(sys.argv[1]), Path(sys.argv[2]), Path(sys.argv[3])
CL = ['Glioma', 'Meningioma', 'No Tumor', 'Pituitary']
sp = pd.read_csv(res / 'metrics' / 'splits.csv').set_index('rel')
def thumb(rel):
    a = np.asarray(Image.open(root / rel).convert('L').resize((64, 64), Image.BILINEAR), dtype=np.float32).ravel()
    a = a - a.mean(); n = np.linalg.norm(a); return a / n if n > 0 else a
X = np.stack([thumb(r) for r in sp.index]); sp['i'] = np.arange(len(sp))
tr = sp[sp.split == 'train']; Xtr = X[tr.i.values]; ytr = np.asarray(tr.label.values, dtype=object)
def style(stem):
    if re.match(r'^(Tr|Te)-', stem): return 'Tr-/Te- prefixed'
    return 'other filename pattern'
rows = []
for split in ['val', 'test']:
    d = sp[sp.split == split]; S = X[d.i.values] @ Xtr.T
    nn = S.argmax(1); mx = S.max(1)
    # same-class best similarity and best other-class similarity
    yd = np.asarray(d.label.values, dtype=object); same = np.where(ytr[None, :] == yd[:, None], S, -1).max(1); other = np.where(ytr[None, :] != yd[:, None], S, -1).max(1)
    r = pd.DataFrame({'rel': d.index, 'split': split, 'label': d.label.values, 'max_corr_train': mx, 'nn_label': ytr[nn], 'nn_correct': ytr[nn] == yd, 'best_same_class': same, 'best_other_class': other, 'style': d.stem.map(style).values})
    rows.append(r)
R = pd.concat(rows); R.to_csv(out / 'tables' / 'nearest_train_neighbour_similarity.csv', index=False)
T = R[R.split == 'test'].set_index('rel')
print('1-nearest-neighbour classification by thumbnail correlation (train as reference):')
for s in ['val', 'test']: print(f'  {s}: accuracy {R[R.split == s].nn_correct.mean():.4f} (n={int((R.split == s).sum())})')
print('\ntest images, nearest-train similarity (Pearson r of 64x64 thumbnails):')
print(T.groupby('style').max_corr_train.describe()[['count', 'mean', '25%', '50%', '75%', 'max']].round(3).to_string())
a, b = T[T['style'] == 'Tr-/Te- prefixed'].max_corr_train, T[T['style'] != 'Tr-/Te- prefixed'].max_corr_train
print(f'  Mann-Whitney p (Tr-/Te- vs other, nearest-train correlation): {mannwhitneyu(a, b).pvalue:.2e}')
print('\n1-NN accuracy on test by filename family:', T.groupby('style').nn_correct.mean().round(3).to_dict())
# do ResNet errors relate to similarity? use images wrong in any of the 6 runs
runs6 = ['R18-C', 'R18-C-s43', 'R18-C-s44', 'R50-C', 'F-R50-s43', 'F-R50-s44']
w = sum((~pd.read_csv(res / 'predictions' / f'{e}_test_predictions.csv').set_index('rel').loc[T.index, 'correct']).astype(int) for e in runs6)
T['n_wrong_6'] = w.values
print('\nmedian nearest-train similarity: images never wrong %.3f | wrong in >=1 of 6 runs %.3f | wrong in >=4 runs %.3f' % (T[T.n_wrong_6 == 0].max_corr_train.median(), T[T.n_wrong_6 >= 1].max_corr_train.median(), T[T.n_wrong_6 >= 4].max_corr_train.median()))
print('  Mann-Whitney p (never wrong vs wrong in >=1):', f'{mannwhitneyu(T[T.n_wrong_6 == 0].max_corr_train, T[T.n_wrong_6 >= 1].max_corr_train).pvalue:.2e}')
T['sim_bin'] = pd.cut(T.max_corr_train, [0, .8, .9, .95, 1.0001], labels=['<0.80', '0.80-0.90', '0.90-0.95', '>=0.95'])
g = T.groupby('sim_bin', observed=True).agg(images=('n_wrong_6', 'size'), mean_err_rate_6_runs=('n_wrong_6', lambda x: x.sum() / (6 * len(x))), wrong_in_any_run=('n_wrong_6', lambda x: int((x > 0).sum())))
print('\nResNet error rate by similarity to the closest training image:'); print(g.round(4).to_string()); g.to_csv(out / 'tables' / 'error_rate_by_nearest_train_similarity.csv')
print('\nshare of test images with a training image at r >= 0.90:', round(float((T.max_corr_train >= .9).mean()), 3), '| by family:', T.groupby('style').apply(lambda d: round(float((d.max_corr_train >= .9).mean()), 3)).to_dict())
# within the family comparison controlling for similarity: errors among images with similarity < 0.90
lo = T[T.max_corr_train < .9]; print('\nimages with nearest-train r < 0.90 only: error rate by family:', lo.groupby('style').apply(lambda d: round(d.n_wrong_6.sum() / (6 * len(d)), 4)).to_dict(), '| n:', lo.groupby('style').size().to_dict())

# ---- figure: error rate vs similarity to nearest training image, split by filename family
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False, 'figure.facecolor': 'white', 'axes.facecolor': 'white'})
C = {'Tr-/Te- prefixed': '#2a78d6', 'other filename pattern': '#eb6834'}
T['hi'] = np.where(T.max_corr_train >= .9, 'r >= 0.90', 'r < 0.90')
fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))
g = T.groupby('sim_bin', observed=True).apply(lambda d: pd.Series({'rate': 100 * d.n_wrong_6.sum() / (6 * len(d)), 'n': len(d)}))
ax[0].bar(range(len(g)), g.rate, color='#2a78d6', width=.6)
for i, (r, n) in enumerate(zip(g.rate, g.n)): ax[0].text(i, r + .1, f'{r:.1f}%\n(n={int(n)})', ha='center', va='bottom', fontsize=10)
ax[0].set_xticks(range(len(g))); ax[0].set_xticklabels(g.index); ax[0].set_ylim(0, g.rate.max() * 1.35)
ax[0].set_xlabel('Highest correlation with any training image'); ax[0].set_ylabel('Error rate, 6 fine-tuned runs pooled (%)'); ax[0].set_title('All test images', loc='left')
x = np.arange(2); w = .36
for k, (fam, col) in enumerate(C.items()):
    vals = []; ns = []
    for h in ['r < 0.90', 'r >= 0.90']:
        d = T[(T['style'] == fam) & (T.hi == h)]; vals.append(100 * d.n_wrong_6.sum() / (6 * max(len(d), 1))); ns.append(len(d))
    ax[1].bar(x + (k - .5) * w, vals, w, color=col, label=fam)
    for xi, v, n in zip(x + (k - .5) * w, vals, ns): ax[1].text(xi, v + .1, f'{v:.1f}%\n(n={n})', ha='center', va='bottom', fontsize=9)
ax[1].set_xticks(x); ax[1].set_xticklabels(['nearest train r < 0.90', 'nearest train r >= 0.90']); ax[1].set_ylim(0, 12.5)
ax[1].set_title('By filename family', loc='left'); ax[1].legend(loc='upper right')
fig.tight_layout(); (out / 'figures').mkdir(exist_ok=True); fig.savefig(out / 'figures' / 'error_rate_by_nearest_train_similarity.png', dpi=200, bbox_inches='tight')
fam = T.groupby(['style', 'hi']).apply(lambda d: pd.Series({'images': len(d), 'error_rate_6_runs': d.n_wrong_6.sum() / (6 * len(d))})).round(4)
fam.to_csv(out / 'tables' / 'error_rate_by_similarity_and_filename_family.csv'); print('\n', fam.to_string())

In [ ]:
%%writefile /content/stage7/merge_results.py
"""Merge the result folders of the separate Colab sessions into ONE canonical results folder.
Naming: 'R18-C' = Stage 6 execution of seed 42 (used in the 3-seed comparison);
        'R18-C-orig' = the earlier Stage 4 execution of the same configuration and seed (kept, not discarded)."""
import sys, shutil, re
from pathlib import Path
import pandas as pd
sp, out = Path(sys.argv[1]), Path(sys.argv[2])
if out.exists(): shutil.rmtree(out)
for d in ['metrics', 'predictions']: (out / d).mkdir(parents=True)
SRC = {  # run id -> (folder, id inside that folder)
    **{e: ('baseline', e) for e in ['B1', 'B2']},
    **{e: ('r18orig', e) for e in ['R18-A', 'R18-B']}, 'R18-C-orig': ('r18orig', 'R18-C'),
    **{e: ('s6a', e) for e in ['R18-C', 'R18-C-s43', 'R18-C-s44', 'T1-lr1e-5', 'T2-lr1e-3', 'T3-bs16', 'T4-adamw', 'T5-dropout', 'T6-weighted', 'T7-hflip']},
    **{e: ('r50orig', e) for e in ['R50-A', 'R50-B', 'R50-C']}, **{e: ('s6b', e) for e in ['F-R50-s43', 'F-R50-s44']}}
n = 0
for new, (folder, old) in SRC.items():
    for sub in ['metrics', 'predictions']:
        for f in (sp / folder / sub).glob(f'{old}_*'):
            if not re.fullmatch(rf'{re.escape(old)}_(history|val_.*|test_.*)\.(csv|json)', f.name): continue
            shutil.copy2(f, out / sub / f.name.replace(old, new, 1)); n += 1
for f in ['splits.csv', 'stage2_config.json', 'shortcut_diagnostic.csv', 'exact_duplicates_removed.csv', 'near_duplicate_pairs.csv']:
    shutil.copy2(sp / 'r18orig' / 'metrics' / f, out / 'metrics' / f)
t = pd.read_csv(sp / 's6b' / 'metrics' / 'experiment_table.csv')
o = pd.read_csv(sp / 'r18orig' / 'metrics' / 'experiment_table.csv'); o = o[o.experiment_id == 'R18-C'].copy(); o['experiment_id'] = 'R18-C-orig'
t = pd.concat([t, o], ignore_index=True)
order = ['B1', 'B2', 'R18-A', 'R18-B', 'R18-C-orig', 'R18-C', 'R18-C-s43', 'R18-C-s44', 'T1-lr1e-5', 'T2-lr1e-3', 'T3-bs16', 'T4-adamw', 'T5-dropout', 'T6-weighted', 'T7-hflip', 'R50-A', 'R50-B', 'R50-C', 'F-R50-s43', 'F-R50-s44']
t['o'] = t.experiment_id.map({k: i for i, k in enumerate(order)}); assert t.o.notna().all()
t = t.sort_values('o').drop(columns='o')
notes = {'R18-C-orig': 'Stage 4 execution of seed 42 (first run of this configuration)',
         'R18-C': 'Stage 6 execution of seed 42; used in the 3-seed comparison and as the final checkpoint candidate',
         'R50-C': 'Stage 5 execution of seed 42', 'B1': 'main baseline', 'B2': 'baseline with class-weighted loss; reached the 40-epoch limit'}
t['notes'] = t.experiment_id.map(notes).fillna('')
t.to_csv(out / 'metrics' / 'experiment_table_master.csv', index=False)
print('copied', n, 'files;', len(t), 'runs:', t.experiment_id.tolist())
missing = [e for e in order for s in ['val', 'test'] if not (out / 'predictions' / f'{e}_{s}_predictions.csv').exists()]
print('runs without prediction files:', missing or 'none')

In [ ]:
%%writefile /content/stage7/evaluate.py
"""Prediction, metrics, saving of results and the experiment table."""
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix,
                             precision_recall_fscore_support, roc_auc_score, f1_score)


def predict(model, loader, device, criterion=None):
    """Run the model on a loader WITHOUT shuffling. Returns labels, predictions, softmax probabilities,
    optional mean loss, and the relative file path of every image (same order)."""
    import torch
    model.eval()
    ys, logits, loss_sum, n = [], [], 0.0, 0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device, non_blocking=True); y = y.to(device, non_blocking=True)
            out = model(x).float()
            if criterion is not None:
                loss_sum += criterion(out, y).item() * x.size(0)
            n += x.size(0); logits.append(out.cpu()); ys.append(y.cpu())
    probs = torch.softmax(torch.cat(logits), dim=1).numpy().astype(np.float64)
    y_true = torch.cat(ys).numpy()
    rels = loader.dataset.df.rel.values
    assert len(rels) == len(y_true), 'loader order / length mismatch'
    return dict(y_true=y_true, y_pred=probs.argmax(1), probs=probs,
                loss=(loss_sum / n) if criterion is not None else None, rels=rels)


def compute_metrics(y_true, probs, class_names):
    """All metrics are computed from the labels and probabilities passed in."""
    K = len(class_names); labels = list(range(K))
    y_pred = probs.argmax(1)
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
    mp, mr, mf, _ = precision_recall_fscore_support(y_true, y_pred, labels=labels, average='macro', zero_division=0)
    wp, wr, wf, _ = precision_recall_fscore_support(y_true, y_pred, labels=labels, average='weighted', zero_division=0)
    tp = np.diag(cm); fn = cm.sum(1) - tp; fp = cm.sum(0) - tp; tn = cm.sum() - tp - fn - fp
    sens = tp / np.maximum(tp + fn, 1)      # = recall of the class
    spec = tn / np.maximum(tn + fp, 1)      # one-vs-rest specificity
    pc = pd.DataFrame(dict(cls=class_names, precision=p, recall_sensitivity=r, specificity=spec, f1=f, support=s))
    P = probs / probs.sum(1, keepdims=True)
    try:
        auc_macro = roc_auc_score(y_true, P, multi_class='ovr', average='macro', labels=labels)
        auc_weighted = roc_auc_score(y_true, P, multi_class='ovr', average='weighted', labels=labels)
        auc_pc = [roc_auc_score((y_true == k).astype(int), P[:, k]) for k in labels]
    except ValueError:
        auc_macro = auc_weighted = float('nan'); auc_pc = [float('nan')] * K
    pc['roc_auc_ovr'] = auc_pc
    summary = dict(n=int(len(y_true)), accuracy=accuracy_score(y_true, y_pred),
                   balanced_accuracy=balanced_accuracy_score(y_true, y_pred),
                   precision_macro=mp, recall_macro=mr, f1_macro=mf,
                   precision_weighted=wp, recall_weighted=wr, f1_weighted=wf,
                   roc_auc_ovr_macro=auc_macro, roc_auc_ovr_weighted=auc_weighted)
    return dict(summary={k: float(v) if not isinstance(v, int) else v for k, v in summary.items()},
                per_class=pc, confusion_matrix=cm)


def bootstrap_ci(y_true, y_pred, n_boot=1000, seed=42, alpha=0.05):
    """Percentile bootstrap CI (resampling test images) for accuracy and macro-F1.
    It reflects test-set sampling noise only, not training randomness."""
    rng = np.random.default_rng(seed); n = len(y_true); acc, f1 = [], []
    for _ in range(n_boot):
        i = rng.integers(0, n, n)
        acc.append((y_true[i] == y_pred[i]).mean()); f1.append(f1_score(y_true[i], y_pred[i], average='macro'))
    q = [100 * alpha / 2, 100 * (1 - alpha / 2)]
    return dict(accuracy_ci=tuple(np.percentile(acc, q).round(4)), f1_macro_ci=tuple(np.percentile(f1, q).round(4)))


def evaluate_split(model, loader, device, class_names, tag, out_root, criterion=None, n_boot=1000, seed=42):
    """Predict, compute metrics, and save metrics / per-class table / confusion matrix / predictions."""
    out_root = Path(out_root)
    res = predict(model, loader, device, criterion)
    m = compute_metrics(res['y_true'], res['probs'], class_names)
    m['summary']['loss'] = res['loss']
    m['bootstrap_ci'] = bootstrap_ci(res['y_true'], res['y_pred'], n_boot, seed)
    save_results(tag, res, m, class_names, out_root)
    return m, res


def save_results(tag, res, m, class_names, out_root):
    out_root = Path(out_root)
    (out_root / 'metrics').mkdir(parents=True, exist_ok=True); (out_root / 'predictions').mkdir(parents=True, exist_ok=True)
    json.dump(dict(summary=m['summary'], bootstrap_ci=m.get('bootstrap_ci')), open(out_root / 'metrics' / f'{tag}_metrics.json', 'w'), indent=2, default=str)
    m['per_class'].to_csv(out_root / 'metrics' / f'{tag}_per_class.csv', index=False)
    pd.DataFrame(m['confusion_matrix'], index=class_names, columns=class_names).to_csv(out_root / 'metrics' / f'{tag}_confusion_matrix.csv')
    pred = pd.DataFrame(dict(rel=res['rel'] if 'rel' in res else res['rels'], y_true=res['y_true'], y_pred=res['y_pred']))
    pred['true_label'] = [class_names[i] for i in pred.y_true]; pred['pred_label'] = [class_names[i] for i in pred.y_pred]
    pred['correct'] = pred.y_true == pred.y_pred
    for k, c in enumerate(class_names):
        pred[f'prob_{c}'] = res['probs'][:, k]
    pred.to_csv(out_root / 'predictions' / f'{tag}_predictions.csv', index=False)


EXPERIMENT_COLUMNS = ['experiment_id', 'model', 'learning_rate', 'batch_size', 'optimizer', 'weight_decay', 'loss',
                      'max_epochs', 'best_epoch', 'epochs_run', 'finetune_strategy', 'augmentation',
                      'trainable_params', 'val_accuracy', 'val_macro_f1', 'test_accuracy', 'test_macro_f1',
                      'roc_auc_test_ovr_macro', 'train_time_s', 'seed']


def log_experiment(path, row):
    """Add or replace one row (by experiment_id) in the experiment table. Only measured values go in."""
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(path) if path.exists() else pd.DataFrame(columns=EXPERIMENT_COLUMNS)
    df = df[df.experiment_id != row['experiment_id']]
    df = pd.concat([df, pd.DataFrame([row]).reindex(columns=EXPERIMENT_COLUMNS)], ignore_index=True)
    df.to_csv(path, index=False)
    return df

In [ ]:
%%writefile /content/stage7/visualization.py
"""Report-ready plots (matplotlib / seaborn only)."""
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc


def _save(fig, path):
    if path is not None:
        fig.savefig(path, dpi=200, bbox_inches='tight')


def plot_history(hist, title, path=None, best_epoch=None):
    """Loss curves (train/val) and accuracy curves (train/val), one panel each."""
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(hist.epoch, hist.train_loss, label='Training loss'); ax[0].plot(hist.epoch, hist.val_loss, label='Validation loss')
    ax[1].plot(hist.epoch, hist.train_acc, label='Training accuracy (augmented batches)'); ax[1].plot(hist.epoch, hist.val_acc, label='Validation accuracy')
    ax[1].plot(hist.epoch, hist.val_macro_f1, '--', label='Validation macro F1')
    for a, t, yl in zip(ax, ['Loss', 'Accuracy / F1'], ['Cross-entropy loss', 'Score']):
        if best_epoch: a.axvline(best_epoch, color='gray', ls=':', label=f'Best epoch ({best_epoch})')
        a.set_xlabel('Epoch'); a.set_ylabel(yl); a.set_title(t); a.grid(alpha=.3); a.legend(fontsize=8)
    fig.suptitle(title); fig.tight_layout(); _save(fig, path); return fig


def plot_confusion_matrix(cm, class_names, title, path=None):
    """Counts (left) and row-normalised percentages = recall per class (right)."""
    cm = np.asarray(cm); norm = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=ax[0], cbar=False)
    sns.heatmap(norm * 100, annot=True, fmt='.1f', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=ax[1], cbar=False, vmin=0, vmax=100)
    ax[0].set_title('Counts'); ax[1].set_title('Row-normalised (% of true class)')
    for a in ax: a.set_xlabel('Predicted'); a.set_ylabel('True'); a.tick_params(axis='x', rotation=20)
    fig.suptitle(title); fig.tight_layout(); _save(fig, path); return fig


def plot_roc(y_true, probs, class_names, title, path=None):
    """One-vs-rest ROC curve for every class."""
    fig, ax = plt.subplots(figsize=(5.5, 5))
    for k, c in enumerate(class_names):
        fpr, tpr, _ = roc_curve((y_true == k).astype(int), probs[:, k])
        ax.plot(fpr, tpr, label=f'{c} (AUC = {auc(fpr, tpr):.3f})')
    ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Chance')
    ax.set_xlabel('False positive rate'); ax.set_ylabel('True positive rate'); ax.set_title(title); ax.legend(fontsize=8, loc='lower right'); ax.grid(alpha=.3)
    fig.tight_layout(); _save(fig, path); return fig


def plot_per_class(per_class, title, path=None):
    """Grouped bars: precision, recall, F1 for every class."""
    d = per_class.melt(id_vars='cls', value_vars=['precision', 'recall_sensitivity', 'f1'], var_name='metric', value_name='value')
    d['metric'] = d.metric.replace({'recall_sensitivity': 'recall'})
    fig, ax = plt.subplots(figsize=(7.5, 4))
    sns.barplot(data=d, x='cls', y='value', hue='metric', ax=ax)
    for c in ax.containers: ax.bar_label(c, fmt='%.2f', fontsize=7)
    ax.set_ylim(0, 1.08); ax.set_xlabel('Class'); ax.set_ylabel('Score'); ax.set_title(title); ax.legend(title='')
    fig.tight_layout(); _save(fig, path); return fig


def plot_experiment_bars(table, path=None, title='Validation and test macro F1 by experiment'):
    """Grouped bars (validation vs test macro F1) for every row of the experiment table."""
    d = table.melt(id_vars='experiment_id', value_vars=['val_macro_f1', 'test_macro_f1'], var_name='split', value_name='macro_f1')
    d['split'] = d.split.map({'val_macro_f1': 'validation', 'test_macro_f1': 'test'})
    fig, ax = plt.subplots(figsize=(max(6, 1.3 * len(table) + 2), 4))
    sns.barplot(data=d, x='experiment_id', y='macro_f1', hue='split', ax=ax)
    for c in ax.containers: ax.bar_label(c, fmt='%.3f', fontsize=7)
    lo = max(0.0, float(d.macro_f1.min()) - 0.1)
    ax.set_ylim(lo, 1.02); ax.set_xlabel('Experiment'); ax.set_ylabel('Macro F1'); ax.set_title(title); ax.legend(title='')
    fig.tight_layout(); _save(fig, path); return fig

## 2. Load the merged results

In [ ]:
RES = Path('/content/final_results')
if not (RES/'metrics'/'experiment_table_master.csv').exists():
    from google.colab import files
    print('Upload final_results.zip'); up = files.upload()
    with zipfile.ZipFile(list(up)[0]) as f: f.extractall(RES)
    # the zip may contain a top-level folder
    if not (RES/'metrics').exists():
        sub = [p for p in RES.iterdir() if (p/'metrics').exists()]; assert sub, 'metrics/ folder not found'; RES = sub[0]
print('Results folder:', RES, '| files:', sum(1 for _ in RES.rglob('*') if _.is_file()))

## 3. Dataset

In [ ]:
DATA_SOURCE = 'kaggle'                                         # 'kaggle' or 'drive'
DRIVE_ZIP = '/content/drive/MyDrive/brain_tumor_dataset.zip'   # only used if DATA_SOURCE == 'drive'
SLUG = 'ahmedsorour1/mri-for-brain-tumor-with-bounding-boxes'
RAW = Path('/content/data_raw'); RAW.mkdir(exist_ok=True)
if not any(p.is_file() for p in RAW.rglob('*')):
    if DATA_SOURCE == 'drive':
        from google.colab import drive; drive.mount('/content/drive')
        assert Path(DRIVE_ZIP).exists(), f'Not found: {DRIVE_ZIP}'
        with zipfile.ZipFile(DRIVE_ZIP) as f: f.extractall(RAW)
    else:
        if not (os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY')):
            try:
                from google.colab import userdata
                os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME'); os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
            except Exception:
                from google.colab import files
                print('Upload kaggle.json'); up = files.upload()
                kd = Path.home()/'.kaggle'; kd.mkdir(exist_ok=True); (kd/'kaggle.json').write_bytes(up[list(up)[0]]); os.chmod(kd/'kaggle.json', 0o600)
        subprocess.run(['pip','-q','install','kaggle'], check=True)
        r = subprocess.run(['kaggle','datasets','download','-d',SLUG,'-p',str(RAW),'--unzip'], capture_output=True, text=True)
        print(r.stdout[-500:], r.stderr[-800:]); r.check_returncode()
def find_root(base):
    for p in [base] + [q for q in base.rglob('*') if q.is_dir()]:
        if {'train','val'} <= {c.name.lower() for c in p.iterdir() if c.is_dir()}: return p
    raise FileNotFoundError('Train/ and Val/ not found')
ROOT = find_root(RAW); print('Dataset root:', ROOT)

## 4. Final evaluation: tables, figures, error analysis

In [ ]:
OUT = Path('/content/stage7_out'); OUT.mkdir(exist_ok=True)
!python /content/stage7/final_evaluation.py --results {RES} --data-root {ROOT} --src /content/stage7 --out {OUT}

## 5. Similarity to the nearest training image
The dataset has no patient identifiers. This check asks whether test errors are related to how similar an image is to its closest training image (64 by 64 grayscale correlation) and whether the pattern differs by filename family.

In [ ]:
!python /content/stage7/neighbour_check.py {RES} {ROOT} {OUT}

## 6. Look at the main figures

In [ ]:
from IPython.display import Image, display
for f in ['model_comparison_macro_f1','confusion_matrix_final_vs_baseline','error_rate_by_factor','error_rate_by_nearest_train_similarity']:
    p = OUT/'figures'/f'{f}.png'
    if p.exists(): display(Image(str(p), width=900))
import pandas as pd; pd.set_option('display.width', 200)
print(pd.read_csv(OUT/'tables'/'model_comparison_table.csv').to_string(index=False))

## 7. Package

In [ ]:
import shutil
shutil.make_archive('/content/stage7_results', 'zip', OUT)
try:
    from google.colab import files; files.download('/content/stage7_results.zip')
except Exception: print('zip at /content/stage7_results.zip')